# Stage 3: Train YOLO26-Pico-P2 with COCO Pretrained Stem Transfer (`03_train_pretrained_backbone.ipynb`)

### Experiment 3: `exp3_pretrained` (Model Registry `m003`)

This notebook trains our custom <500k parameter detector initialized with **pretrained COCO stem/backbone weights** transferred from `yolo26n.pt` into layers 0..4 (stem convolutions and C3k2 feature extractors).

Key constraints:
- **Rule 1 (Data Split Isolation)**: Video-level isolation strictly maintained. `eval` and `eval_B` are isolated in `test`.
- **Rule 4 (Optical Distance Stratification)**: Evaluation across `0–200 m` (near) and `200–400 m` (far).
- **Rule 5 (Parameter Constraint)**: < 500,000 parameters strictly enforced.

In [ ]:
# === CELL 1: GPU & REPOSITORY SETUP ===
import os
import sys
from pathlib import Path
import torch

GITHUB_REPO_URL = "https://github.com/ituvtu/aerial-vehicle-detector.git"
REPO_DIR = Path("/content/aerial-vehicle-detector")

if not REPO_DIR.exists():
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only origin main

%cd {REPO_DIR}

!pip install -q "ultralytics>=8.4.0" "git+https://github.com/ultralytics/CLIP.git" "pandas" "pyyaml" "matplotlib"

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    !nvidia-smi

In [ ]:
# === CELL 2: DATASET SYNCHRONIZATION ===
import yaml
from src.manifest_manager import ensure_images_available, ensure_raw_eval_video_available, sync_manifest_splits

with open("configs/project_config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

ensure_images_available(
    manifest_path=cfg["paths"]["manifest_path"],
    images_dir=cfg["paths"]["images_dir"],
    bundle_zip_path=cfg["paths"]["images_bundle_zip"],
    bundle_url=cfg.get("remote_data", {}).get("images_bundle_url"),
)
split_counts = sync_manifest_splits()
print("Frames unpacked. Dataset split distribution:", split_counts)

# Ensure raw 30 FPS evaluation video is downloaded for full-fidelity evaluation demo
raw_eval_video = ensure_raw_eval_video_available("32179597")
print(f"Raw evaluation video ready: {raw_eval_video} ({raw_eval_video.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# === CELL 3: ARCHITECTURE & PRETRAINED STEM TRANSFER VERIFICATION ===
from ultralytics import YOLO
from src.metrics import inspect_and_verify_architecture, transfer_coco_stem_weights

# Verify architecture parameter limits (<500k params, Rule 5)
arch_report = inspect_and_verify_architecture(
    yaml_path="configs/yolo26_pico_p2.yaml",
    pretrained_weights="yolo26n.pt",
    output_report_path="results/model_architecture_report.json",
    max_params=500_000,
)

print(f"\nArchitecture Verified: {arch_report['unfused_params']:,} unfused / {arch_report['fused_params']:,} fused params")

# Test deterministic PyTorch stem transfer into YOLO26-Pico-P2
test_model = YOLO("configs/yolo26_pico_p2.yaml")
transfer_info = transfer_coco_stem_weights(test_model, weights_path="yolo26n.pt")
print("Transfer Verification:", transfer_info)

In [ ]:
# === CELL 4: TRAINING (exp3_pretrained) ===
# Trains YOLO26-Pico-P2 with transferred stem weights on 73 train frames
!python scripts/04_train.py --mode exp3_pretrained --pretrained-stem --epochs 150 --batch 8 --imgsz 1024

In [ ]:
# === CELL 5: TRAINING TELEMETRY & VALIDATION CALIBRATION ===
from IPython.display import Image, display
from pathlib import Path

# Show registered models in ledger
!python scripts/04_train.py --list-models

curves_p = Path("results/training_curves_exp3_pretrained.png")
if curves_p.exists():
    display(Image(filename=str(curves_p)))

In [ ]:
# === CELL 6: INDEPENDENT TEST EVALUATION (m003) ===
# Decoupled evaluation on held-out test split (eval + eval_B)
!python scripts/05_evaluate.py --weights weights/exp3_pretrained_best.pt --calibration weights/calibration_val_exp3_pretrained.json

from IPython.display import Markdown, display
from pathlib import Path

eval_summary_p = Path("results/eval_summary_table.md")
if eval_summary_p.exists():
    display(Markdown(eval_summary_p.read_text(encoding="utf-8")))

In [ ]:
# === CELL 7: 30 FPS FULL-HD EVALUATION DEMO VIDEO ===
!python scripts/05_evaluate.py --weights weights/exp3_pretrained_best.pt --calibration weights/calibration_val_exp3_pretrained.json

demo_video_p = Path("results/eval_demo_32179597_full_fps.mp4")
if demo_video_p.exists():
    print(f"\nFull 30.0 FPS Evaluation Video generated: {demo_video_p} ({demo_video_p.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# === CELL 8: PACKAGE & DOWNLOAD DELIVERABLES BUNDLE ===
import datetime
import zipfile
from pathlib import Path

timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
bundle_zip = Path(f"colab_results_bundle_m003_{timestamp}.zip")

include_dirs = [Path("weights"), Path("results"), Path("assets"), Path("data/registry")]

with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in include_dirs:
        if folder.exists():
            for item in folder.rglob("*"):
                if item.is_file() and not item.name.endswith(".mp4") and item.stat().st_size < 50_000_000:
                    zf.write(item, arcname=item.as_posix())

print(f"Packaged deliverables bundle: {bundle_zip} ({bundle_zip.stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files
    files.download(str(bundle_zip))
    print("Download triggered in browser.")
except (ImportError, Exception) as exc:
    print(f"Note: Running locally or download skipped: {exc}. Bundle saved at {bundle_zip}")